In [ ]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
import polars
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
#plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import xgboost as xgb
import skforecast
import sklearn
from xgboost import XGBRegressor
from sklearn.feature_selection import RFECV
from skforecast.ForecasterAutoreg import ForecasterAutoreg
from skforecast.model_selection import bayesian_search_forecaster
from skforecast.model_selection import backtesting_forecaster
from skforecast.model_selection import select_features
import shap

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

from sklearn.preprocessing import LabelEncoder

In [ ]:
xgdf = pd.read_parquet('C:/Users/J.Heuvelmans/OneDrive - Brain Research Center/Documenten/EAISI/2024Supermarket/Data/processed/Aggregated_data_20241106.parquet', engine='pyarrow')

xgdf.head()

In [ ]:
#Data transformations
xgdf['week_number'] = xgdf['date'].dt.isocalendar().week
xgdf['month'] = xgdf['date'].dt.month
xgdf['year'] = xgdf['date'].dt.year

xgdf['unique_id'] = xgdf['store_nbr'].astype(str) + "_" + xgdf['item_nbr'].astype(str)

# Encode `store_id`
le = LabelEncoder()
xgdf['unique_id'] = le.fit_transform(xgdf['unique_id'])

#Sort timeseries by unique id and date
xgdf=xgdf.sort_values(by=['unique_id','date'])

for i in range(1, 53):
    # Create the new column dynamically
    col_name = f'last{7 * i}Value'
    xgdf[col_name] = xgdf.groupby('unique_id')['unit_sales'].shift(i)

xgdf.head(100)

In [ ]:
#xgdf = pd.get_dummies(xgdf, columns=['store_cluster'])
#xgdf = pd.get_dummies(xgdf, columns=['store_nbr'])
#xgdf = pd.get_dummies(xgdf, columns=['item_class'])
#xgdf = pd.get_dummies(xgdf, columns=['item_family'])
xgdf = pd.get_dummies(xgdf, columns=['store_type'])
xgdf

In [ ]:
def normalize(series):
    # Ensure the series is numeric (convert to float)
    series = series.astype(float)  # Convert to float directly
    min_val = series.min()
    max_val = series.max()
    return (series - min_val) / (max_val - min_val)

col_to_normalize = ['week_number', 'year', 'month']

n_xgdf = xgdf.copy() 

for column in col_to_normalize:
    n_xgdf[column] = normalize(n_xgdf[column])

In [ ]:
# Ensure 'date' is of datetime type for comparison
n_xgdf['date'] = pd.to_datetime(n_xgdf['date'])

df = n_xgdf

# 1. Filter data based on the 'date' range
df_train = df[(df['week_number_cum'] > 138) & (df['week_number_cum'] <= 190)]
df_test = df[(df['week_number_cum'] > 190) & (df['week_number_cum'] <= 216)]
df_validate = df[df['week_number_cum'] > 216]

# 2. Sort by 'date'
df_train = df_train.sort_values(by='week_number_cum')
df_test = df_test.sort_values(by='week_number_cum')
df_validate = df_validate.sort_values(by='week_number_cum')

# 3. Drop 'unit_sales' and 'date' columns for X_train and X_test
X_train = df_train.drop(columns=['unit_sales', 'date', 'last7Value', 'item_family'])
X_test = df_test.drop(columns=['unit_sales', 'date', 'last7Value', 'item_family'])

# 4. Rename 'unit_sales' to 'Label' for y_train and y_test
y_train = df_train[['unit_sales']].rename(columns={'unit_sales': 'Label'})
y_test = df_test[['unit_sales']].rename(columns={'unit_sales': 'Label'})



In [ ]:
#print(xgdf.columns)

#n_xgdf.head()
categorical_columns = xgdf.select_dtypes(include=['category']).columns
print(categorical_columns)


In [ ]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred[y_pred < 0] = 0

a = y_test.rename(columns={'Label': 'y'})
b=pd.DataFrame(y_pred, columns=['y_p'])



In [ ]:
print(a)

In [ ]:
print(b)

In [ ]:
df_pred = df_test.copy()
df_pred = df_pred[['store_nbr', 'item_nbr', 'unit_sales', 'week_number_cum', 'date', 'unique_id', 'last14Value', 'store_cluster', 'item_family', 'item_class', 'perishable']]
df_pred['y_xgb'] = y_pred

df_pred = df_pred.rename(columns={'last14Value': 'y_naive'})

df_pred['bias_xgb'] = df_pred['unit_sales'] - df_pred['y_xgb']
df_pred['bias_naive'] = df_pred['unit_sales'] - df_pred['y_naive']

df_pred['acc_xgb'] = np.where((df_pred['unit_sales'] == 0) & (df_pred['y_xgb'] != 0), np.nan, (1 - np.abs(df_pred['bias_xgb']) / df_pred['unit_sales']) * 100) 
df_pred['acc_naive'] = np.where((df_pred['unit_sales'] == 0) & (df_pred['y_naive'] != 0), np.nan,(1 - np.abs(df_pred['bias_naive']) / df_pred['unit_sales']) * 100)

df_pred=df_pred.sort_values(by=['unique_id','date'])
df_pred

In [ ]:
def calculate_statistics(df):
    # Calculate means for unit sales and predictions
    mean_a = np.mean(df['unit_sales'])
    mean_p = np.mean(df['y_xgb'])
    mean_n = np.mean(df['y_naive'])
    
    # Calculate mean accuracy for XGB and Naive models
    mean_accuracy = (1 - (np.abs(mean_a - mean_p) / mean_a)) * 100
    accuracy = np.mean(df['acc_xgb'])
    
    mean_accuracy_n = (1 - (np.abs(mean_a - mean_n) / mean_a)) * 100
    accuracy_n = np.mean(df['acc_naive'])
    
    # Calculate standard deviations of accuracy
    sd_acc = np.std(df['acc_xgb'])
    sd_acc_n = np.std(df['acc_naive'])
    
    # Calculate bias and positive/negative bias for XGB and Naive models
    bias = np.mean(df['bias_xgb'])
    positive_bias = np.mean(df['bias_xgb'][df['bias_xgb'] > 0])
    negative_bias = np.mean(df['bias_xgb'][df['bias_xgb'] < 0])
    
    bias_n = np.mean(df['bias_naive'])
    positive_bias_n = np.mean(df['bias_naive'][df['bias_naive'] > 0])
    negative_bias_n = np.mean(df['bias_naive'][df['bias_naive'] < 0])
    
    # Calculate standard deviations of bias
    sd_bias = np.std(df['bias_xgb'])
    sd_bias_n = np.std(df['bias_naive'])
    
    # Store results in a dictionary
    results = {
        'Metric': [
            'Mean Accuracy', 'Accuracy', 'Mean Accuracy Naive', 'Accuracy Naive',
            'Standard Deviation of Accuracy', 'Standard Deviation of Accuracy Naive',
            'Bias', 'Positive Bias', 'Negative Bias', 
            'Bias Naive', 'Positive Bias Naive', 'Negative Bias Naive',
            'Standard Deviation of Bias', 'Standard Deviation of Bias Naive'
        ],
        'Value': [
            mean_accuracy, accuracy, mean_accuracy_n, accuracy_n,
            sd_acc, sd_acc_n,
            bias, positive_bias, negative_bias,
            bias_n, positive_bias_n, negative_bias_n,
            sd_bias, sd_bias_n
        ]
    }
    
    # Convert results dictionary to a DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df

def calculate_metrics_grouped(df, column):
    store_stats = []
    
    # Iterate over each store's data
    for store, group in df.groupby(column):
        # Calculate statistics for the current store using the existing function
        df_metrics = calculate_statistics(group)
        
        # Add the store number to each metric's result for reference
        df_metrics[column] = store
        
        # Append each store's result to the list
        store_stats.append(df_metrics)
    
    # Concatenate all store statistics into a single DataFrame
    store_means = pd.concat(store_stats).reset_index(drop=True)

    # Pivot the DataFrame to make store_nbr columns and Metric rows
    store_metrics_pivot = store_means.pivot(index='Metric', columns=column, values='Value')

    return store_metrics_pivot


def plot_bias_comparison(df):
    """
    Plots a comparison of Negative Bias, Negative Bias Naive, Positive Bias, and Positive Bias Naive
    across stores from a given DataFrame.
    
    Parameters:
    df (pd.DataFrame): DataFrame where rows represent metrics ('Negative Bias', 'Negative Bias Naive',
                        'Positive Bias', 'Positive Bias Naive') and columns are store numbers.
    """
    # Extract the data to plot
    stores = list(df.columns)  # Get store numbers as list of columns excluding the 'Metric' column
    negative_bias = df.loc['Negative Bias', stores].values.astype(float)  # Negative Bias values
    negative_bias_naive = df.loc['Negative Bias Naive', stores].values.astype(float)  # Negative Bias Naive values
    positive_bias = df.loc['Positive Bias', stores].values.astype(float)  # Positive Bias values
    positive_bias_naive = df.loc['Positive Bias Naive', stores].values.astype(float)  # Positive Bias Naive values

    # Set up the plot
    x = np.arange(len(stores))  # Store positions for the x-axis
    width = 0.2  # Width of the bars

    fig, ax = plt.subplots(figsize=(15, 8))

    # Plot the bars for each category
    ax.bar(x - width, negative_bias, width, label='Negative Bias')
    ax.bar(x, negative_bias_naive, width, label='Negative Bias Naive')
    ax.bar(x + width, positive_bias, width, label='Positive Bias')
    ax.bar(x + 2 * width, positive_bias_naive, width, label='Positive Bias Naive')

    # Add labels and formatting
    ax.set_xlabel('Stores')
    ax.set_ylabel('Bias Value')
    ax.set_title('Comparison of Negative and Positive Bias Across Stores')
    ax.set_xticks(x)
    ax.set_xticklabels(stores, rotation=90)
    ax.legend()

    # Display the plot
    plt.tight_layout()
    plt.show()

# Usage example:
# plot_bias_comparison(df)



In [ ]:
df_metrics = calculate_statistics(df_pred)
df_metrics

In [ ]:
df_metrics_stores = calculate_metrics_grouped(df_pred, 'store_nbr')

df_metrics_stores

In [ ]:
plot_bias_comparison(df_metrics_stores)

In [ ]:
df_metrics_perishable = calculate_metrics_grouped(df_pred, 'perishable')

df_metrics_perishable

In [ ]:
# Group by 'store_nbr' and calculate the mean for each column
store_means = df_pred.groupby('store_nbr').mean().reset_index()

store_means

In [ ]:
# Group by 'item_nbr' and calculate the mean for each column
item_means = df_pred.groupby('item_nbr').mean().reset_index()

item_means

In [ ]:
#grouped_df = df_pred.groupby('week_number_cum')[['y_pred', 'unit_sales']].sum().reset_index()
grouped_df = df_pred[(df_pred['item_nbr'] == 103520) & (df_pred['store_nbr'] == 10)]

# Step 2: Plot the results
plt.figure(figsize=(12, 6))
plt.plot(grouped_df['week_number_cum'], grouped_df['y_xgb'], marker='o', label='Predicted Sales (y_pred)', color='blue')
plt.plot(grouped_df['week_number_cum'], grouped_df['unit_sales'], marker='o', label='Actual Sales (unit_sales)', color='orange')
plt.plot(grouped_df['week_number_cum'], grouped_df['y_naive'], marker='o', label='Naive prediction', color='red')

# Add titles and labels
plt.title('Sum of y_pred and Unit Sales Over Weeks')
plt.xlabel('Week Number Cumulative')
plt.ylabel('Sales')
plt.xticks(grouped_df['week_number_cum'])  # Set x-ticks to show all weeks
plt.legend()
plt.grid()

# Show the plot
plt.show()